# 02 · Store

*Part 1, section 2.4 and Part 2, section 2.* Traces are the raw signal; the **evaluation dataset**
is the curated one, and it doubles as the regression suite. It grows from two places: production
failures that a person labeled with the right answer, and hand-written **golden cases** that cover
behavior that has not failed yet.

Two properties make it trustworthy later:

* **provenance**: every row says where it came from (`source`, `source_trace`, `author`, `theme`)
* **a holdout**: a slice no optimizer, analyst or approver ever sees, only the gate

In [ ]:
import _nb as nb

experiment_id = nb.setup()

In [ ]:
import mlflow
from mlflow.genai.datasets import create_dataset
from selfimprove import config, store, traces

dataset = create_dataset(name=config.settings.dataset_name, experiment_id=[experiment_id],
                         tags={"purpose": "regression"})
dataset.name

**Under the hood:** the harness's `store.dataset()` fetches the dataset by name and creates it on
first use, with the tags `purpose` and `holdout_fraction`. This cell made it with `purpose` only;
`approve` adds the rest in notebook 04.

**MLflow in this step**

* `create_dataset(name=..., experiment_id=[...], tags=...)` (from `mlflow.genai.datasets`): an
  evaluation dataset, an entity in the tracking store with an id (`d-...`), linked to the experiment,
  with tags
* `get_dataset(name=...)`: fetches it again on every later call

In [ ]:
from mlflow.genai.datasets import get_dataset

ds = get_dataset(name=config.settings.dataset_name)
print(ds.dataset_id, "| experiments", ds.experiment_ids)
print("tags", {k: v for k, v in ds.tags.items() if not k.startswith("mlflow.")})

## 1. Failures with an expectation become records

Only **human** failures that carry an expectation qualify. A judge verdict never becomes an
expectation: a person or a golden case is the source of truth for what correct means.

In [ ]:
labeled_fails = [
    t for t in traces.production(f"{traces.FAILED} AND {traces.HAS_EXPECTATION}")
    if (fb := traces.human_feedback(t)) is not None and traces.is_fail(fb.value)
]
print(len(labeled_fails), "human-labeled failures with an expected response")

rec = store.record_from_trace(labeled_fails[0], source="human_fail", author="you")
rec

**Under the hood:** `store.record_from_trace()` writes nothing. It returns a plain dict in the shape
`merge_records()` takes: `inputs`, `expectations`, and `tags` for provenance and the split.

**MLflow in this step**

* `trace.data.request`: the trace's input
* `trace.info.assessments`: its expectations (the `expected_response` a reviewer logged)
* record tags: MLflow stores the provenance and split tags on each dataset record

The split is chosen by a **hash of the input**, so a case lands in the same split forever, however
many times it fails:

In [ ]:
for q in ["How long can I wait to request a refund?", "Can I pay in Canadian dollars?",
          "How many teammates can I invite on Pro?"]:
    print(f"{store.split_for(q):<8} {q}")

## 2. Forty complaints, one test

The same question failing many times is one bug. Deduplicate before merging, or the suite bloats
instead of improving (Part 2, section 5).

In [ ]:
records, seen = [], []
for t in labeled_fails:
    r = store.record_from_trace(t, source="human_fail", author="you")
    if store.near_duplicate(store.question_of(r), seen) is None:
        records.append(r)
        seen.append(r)
print(f"{len(labeled_fails)} failures -> {len(records)} distinct tests")
dataset.merge_records(records)

**Under the hood:** the cell deduplicates first because MLflow only recognizes exact repeats. Merging
the same failure twice does not grow the suite, but two different phrasings of one question would.

**MLflow in this step**

* `dataset.merge_records(records)`: an upsert. MLflow hashes each record's `inputs`; a record whose
  inputs already exist is merged into the existing row (its expectations and tags are updated)
  instead of being added again

## 3. Golden cases

Written by a person, including the **holdout** cases (`holdout: true`). These are the cases that
tell you whether an improvement generalizes or just memorized the failures.

In [ ]:
print(open("examples/golden.yaml").read()[:900])

In [ ]:
print("+", store.seed_golden(), "golden cases")
print(store.summary())

**Under the hood:** `store.seed_golden()` reads `golden.yaml`, builds one record per case with
`source=golden`, `author=human` and the split from `holdout:`, and skips inputs that are already in
the dataset.

**MLflow in this step**

* `dataset.merge_records(records)`: adds the golden cases
* `get_dataset(...).to_df()`: how `store.summary()` reads the rows back

Each row is a dataset record with its own id; the split and source live in the record's tags:

In [ ]:
import pandas as pd

df = get_dataset(name=config.settings.dataset_name).to_df()
print(len(df), "records; columns:", list(df.columns))
pd.DataFrame(list(df["tags"])).groupby(["split", "source"]).size()

In [ ]:
df = dataset.to_df()
df.assign(question=df["inputs"].map(lambda i: i["question"][:50]),
          split=df["tags"].map(lambda t: t.get("split")),
          source=df["tags"].map(lambda t: t.get("source")))[["question", "split", "source"]]

Treat the dataset as code: every row carries its provenance (`source`, `author`, `source_trace`),
and every evaluation run logs a digest of the rows it evaluated, so a pass or fail can be traced to
a specific set of cases.

**Next:** [03 · Update and gate](03_update_and_gate.ipynb): the first manual pass of the loop.